In [ ]:
JOBS_GPU0 = ['icarl_full__nobn__superclass__s0', 'icarl_full__nobn__superclass__s2', 'icarl__nobn__superclass__s2']
JOBS_GPU1 = ['icarl_full__nobn__superclass__s1', 'icarl__nobn__superclass__s0', 'icarl__nobn__superclass__s1']
JOBS_CPU = []
RUN_TAG = 'vna_v4_main'
PRESET = 'M1'
REPO_COMMIT = 'd1402e839992db8981253a42c74034a4e6f325e0'
SOURCE_DIGEST = None
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000


# VNA v4 main run
Attach previous output archives to resume. M1/M2 own backbone schedules; C3/C4 require C1 selections.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna4_main")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs')
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from scripts.collect_parallel import validate_result
ROOT = STAGE_B / RUN_TAG
ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')

def attached_file(filename, destination, predicate):
    if destination.exists(): return destination
    candidates = []
    for path in Path('/kaggle/input').rglob(filename):
        try: value = json.loads(path.read_text())
        except (ValueError, OSError): continue
        if predicate(value): candidates.append((path,value))
    if not candidates: return None
    if any(value != candidates[0][1] for _,value in candidates): raise ValueError(f'conflicting {filename}')
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0][0], destination)
    return destination

def completed(job):
    directory = ROOT/job
    if job.endswith('__sweep'):
        path = directory/'best_config.json'
        if not path.exists(): return False
        value = json.loads(path.read_text())
        if value.get('code_commit') != REPO_COMMIT or value.get('source_digest') != SOURCE_DIGEST or value.get('synthetic'):
            raise ValueError('best_config identity mismatch')
        return True
    path = directory/'summary.json'
    if not path.exists(): return False
    summary = json.loads(path.read_text())
    if job.startswith('lrprobe__'): return summary.get('status') in ('PROBE','DIVERGED')
    if summary.get('status') != 'COMPLETED': return False
    protocol = json.loads((directory/'protocol.json').read_text())
    tasks = json.loads((directory/'tasks.json').read_text())
    validate_result(job, summary, protocol, tasks, REPO_COMMIT, SOURCE_DIGEST)
    return True

def schedule_name(backbone): return 'icarl_BN.json' if backbone == 'bn' else 'icarl_noBN.json'

def nobn_lr_scale():
    # Registered noBN lr rule: chosen once by the L1 probe notebook, attached to every noBN run.
    destination = ROOT/'_schedules'/'nobn_lr.json'
    attached_file('nobn_lr.json', destination, lambda v: v.get('code_commit') == REPO_COMMIT
                  and v.get('source_digest') == SOURCE_DIGEST and v.get('rule') == 'largest finite 3-task baseline')
    if not destination.exists(): raise FileNotFoundError('Attach the L1 output (nobn_lr.json) before any noBN run')
    return json.loads(destination.read_text())['nobn_lr_scale']

def select_nobn_lr():
    probes = []
    for scale in LR_SCALES:
        path = ROOT/f'lrprobe__s{scale}'/'summary.json'
        summary = json.loads(path.read_text()) if path.exists() else {}
        probes.append(dict(scale=scale, lr=.1*scale, status=summary.get('status'), tasks=summary.get('tasks_run')))
    finite = [p['scale'] for p in probes if p['status'] == 'PROBE' and p['tasks'] == 3]
    if not finite: raise RuntimeError(f'no noBN lr scale is finite for 3 tasks: {probes}')
    value = dict(rule='largest finite 3-task baseline', nobn_lr_scale=max(finite), lr=.1*max(finite),
                 probes=probes, code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)
    (ROOT/'_schedules').mkdir(parents=True, exist_ok=True)
    (ROOT/'_schedules'/'nobn_lr.json').write_text(json.dumps(value, indent=2))
    print('NOBN_LR', json.dumps(value), flush=True)

def prepare_schedule(backbone):
    filename = schedule_name(backbone)
    destination = ROOT/'_schedules'/filename
    attached_file(filename, destination, lambda v: v.get('code_commit') == REPO_COMMIT
                  and v.get('source_digest') == SOURCE_DIGEST and v.get('authoritative')
                  and v.get('backbone') == ('resnet32' if backbone == 'bn' else 'resnet32_nobn'))
    if destination.exists(): return
    cmd = [sys.executable,'-u','-m','experiments.main_cil','--arm','icarl_full',
           '--backbone','resnet32' if backbone == 'bn' else 'resnet32_nobn',
           '--class-order','superclass','--seed','0','--out',str(ROOT/'_preflight'/backbone),
           '--device','cuda:0','--data-dir',str(DATA_ROOT),'--preflight-only',
           '--schedule-output',str(destination),'--schedule-source','self' if PRESET == 'M3' else 'authority']
    if backbone == 'nobn': cmd += ['--nobn-lr-scale',str(nobn_lr_scale())]
    launch(cmd,0,backbone+'_preflight',RUN_ROOT/f'{RUN_TAG}_{backbone}_preflight.log')
    archive_outputs()
    if not destination.exists() and not pause_due(): raise RuntimeError('preflight did not produce K0')

def command(job):
    if job.startswith('b5_'):
        arm,width,seed = re.fullmatch(r'b5_(.+)__width(\d+)__s(\d+)',job).groups()
        cmd = [sys.executable,'-u','-m','experiments.pilot_v3','--setting','regression',
               '--arm','b5_'+arm,'--width',width,'--seed',seed,'--device','cpu',
               '--out',str(ROOT/job),'--resume']
        if arm != 'bp': cmd += ['--schedule',str(ROOT/'_schedules'/f'b5_width{width}.json')]
        return cmd
    if job.startswith('lrprobe__'):
        return [sys.executable,'-u','-m','experiments.main_cil','--arm','icarl','--backbone','resnet32_nobn',
                '--class-order','superclass','--seed','0','--nobn-lr-scale',job.split('__s')[1],
                '--max-tasks','3','--out',str(ROOT/job),'--device','cuda:0','--data-dir',str(DATA_ROOT)]
    if job.endswith('__sweep'):
        return [sys.executable,'-u','-m','experiments.v4_competitors','--method',job.split('__')[0],
                '--mode','sweep','--device','cuda:0','--data-dir',str(DATA_ROOT),
                '--out',str(ROOT/job),'--resume','--nobn-lr-scale',str(nobn_lr_scale())]
    arm,backbone,order,seed = job.split('__'); seed = seed[1:]
    if arm in ('icarl','icarl_full'):
        cmd = [sys.executable,'-u','-m','experiments.main_cil','--arm',arm,
               '--backbone','resnet32' if backbone == 'bn' else 'resnet32_nobn',
               '--class-order',order,'--seed',seed,'--out',str(ROOT/job),
               '--device','cuda:0','--data-dir',str(DATA_ROOT)]
        if arm == 'icarl_full': cmd += ['--schedule',str(ROOT/'_schedules'/schedule_name(backbone))]
        if backbone == 'nobn': cmd += ['--nobn-lr-scale',str(nobn_lr_scale())]
        return cmd
    cmd = [sys.executable,'-u','-m','experiments.v4_competitors','--method',arm,
           '--backbone',backbone,'--order',order,'--seed',seed,'--out',str(ROOT/job),
           '--device','cuda:0','--data-dir',str(DATA_ROOT),'--resume']
    if PRESET in ('C3','C4'):
        destination = ROOT/'_configs'/arm/'best_config.json'
        path = attached_file('best_config.json', destination, lambda v: v.get('method') == arm
                             and v.get('code_commit') == REPO_COMMIT and v.get('source_digest') == SOURCE_DIGEST
                             and v.get('synthetic') is False)
        if path is None: raise FileNotFoundError(f'Attach C1 best_config.json for {arm}')
        cmd += ['--best-config',str(path)]
    if PRESET == 'C5': cmd += ['--reproduction']
    if backbone == 'nobn': cmd += ['--nobn-lr-scale',str(nobn_lr_scale())]
    return cmd

def run_gpu(gpu,jobs):
    for job in jobs:
        if pause_due(): break
        if completed(job): continue
        if not launch(command(job),gpu,job,RUN_ROOT/f'{RUN_TAG}_{job}.log'): break
        archive_outputs()

def run_cpu(job):
    if pause_due() or completed(job): return
    from experiments.vna_resume import launch_logged
    result = launch_logged(command(job),cwd=REPO,env={**ENV,'CUDA_VISIBLE_DEVICES':''},
        log_path=RUN_ROOT/f'{RUN_TAG}_{job}.log',lane=job,job=job,lanes=lane_state,
        paused=session_paused,deadline=VNA_DEADLINE)
    archive_outputs()
    return result

try:
    if PRESET in ('M1','M2','M3'): prepare_schedule('bn' if PRESET == 'M2' else 'nobn')
    if PRESET == 'B5w':
        for width in (5,20,100):
            destination = ROOT/'_schedules'/f'b5_width{width}.json'
            attached_file(destination.name,destination,lambda v: v.get('code_commit') == REPO_COMMIT
                          and v.get('source_digest') == SOURCE_DIGEST and v.get('authoritative') and v.get('width') == width)
            if not destination.exists():
                from experiments.vna_resume import launch_logged
                cmd = [sys.executable,'-u','-m','experiments.pilot_v3','--setting','regression',
                       '--arm','b5_bp_vna_gir','--width',str(width),'--seed','0','--device','cpu',
                       '--out',str(ROOT/'_preflight'/f'width{width}'),'--resume','--preflight-only',
                       '--schedule-output',str(destination)]
                launch_logged(cmd,cwd=REPO,env={**ENV,'CUDA_VISIBLE_DEVICES':''},
                    log_path=RUN_ROOT/f'{RUN_TAG}_width{width}_preflight.log',lane='CPU',job='preflight',
                    lanes=lane_state,paused=session_paused,deadline=VNA_DEADLINE)
            if destination.exists():
                schedule = json.loads(destination.read_text())
                schedule['width'] = width
                destination.write_text(json.dumps(schedule,indent=2))
            if pause_due(): break
    archive_outputs()
    if not pause_due():
        with ThreadPoolExecutor(max_workers=2) as gpu_pool, ThreadPoolExecutor(max_workers=4) as cpu_pool:
            futures=[gpu_pool.submit(run_gpu,0,JOBS_GPU0),gpu_pool.submit(run_gpu,1,JOBS_GPU1)]
            futures += [cpu_pool.submit(run_cpu,job) for job in JOBS_CPU]
            for future in futures: future.result()
        if PRESET == 'L1' and not pause_due(): select_nobn_lr()
finally:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs()


In [ ]:
from scripts.collect_parallel import collect, report
from experiments.main_cil_results import stability_plasticity
from collections import Counter
records = collect([ROOT.parent],RUN_TAG)
if PRESET == 'C1':
    from experiments.v4_competitors import grid
    for method in ('icarl_cbp','icarl_redo','icarl_snp','icarl_upgd'):
        selection = ROOT/(method+'__sweep')/'best_config.json'
        if not selection.exists(): continue
        best = json.loads(selection.read_text())
        directory = selection.parent/'scan'/str(grid(method).index(best['config']))
        record = {key:json.loads((directory/(key+'.json')).read_text())
                  for key in ('summary','protocol','tasks','diagnostics')}
        job = method+'__nobn__superclass__s0'
        validate_result(job,record['summary'],record['protocol'],record['tasks'],REPO_COMMIT,SOURCE_DIGEST)
        records[job] = dict(record,job=job)
if PRESET not in ('B5w','L1'): report(records)
flags = []
for record in records.values():
    events = record.get('interventions',record.get('vna',[]))
    counts = Counter()
    for event in events:
        for flag in ('gir_fallback','warm_reset','rejected'):
            value = event.get(flag,False)
            if flag == 'warm_reset':
                value = value or any(r.get('warm_reset',False) for r in event.get('cg',[]) if isinstance(r,dict))
            counts[flag] += str(value).lower() in ('true','1')
        counts['skip_reason='+str(event.get('skip_reason') or 'none')] += 1
    flags.append(dict(job=record['job'],events=len(events),
                      rates={key:value/max(1,len(events)) for key,value in counts.items()}))
print(json.dumps(flags,indent=2))
(ROOT/'flag_rates.json').write_text(json.dumps(flags,indent=2))
figure = ROOT/'stability_plasticity.svg'
if PRESET == 'B5w':
    import matplotlib.pyplot as plt
    from collections import defaultdict
    import statistics
    groups = defaultdict(list)
    for record in records.values():
        width = re.search(r'__width(\d+)__',record['job']).group(1)
        groups[(record['protocol']['arm'],width)].append(record['summary'])
    from scripts.collect_parallel import markdown, mean_sd
    rows = []
    for (arm,width),summaries in sorted(groups.items()):
        controls = {v['seed']:v for v in groups.get(('b5_bp',width),[])}
        deltas = [v['mean_last50']-controls[v['seed']]['mean_last50'] for v in summaries if v['seed'] in controls]
        rows.append(dict(width=width,arm=arm,seeds=len(summaries),
            online_error=mean_sd([v['mean_last50'] for v in summaries]),
            paired_delta_bp=mean_sd(deltas),seeds_won=sum(v<0 for v in deltas),
            cpu_hours=mean_sd([v['seconds']/3600 for v in summaries])))
    markdown('B5 widths',rows,['width','arm','seeds','online_error','paired_delta_bp','seeds_won','cpu_hours'])
    fig,ax = plt.subplots()
    for (arm,width),summaries in sorted(groups.items()):
        xs = [v['mean_last50'] for v in summaries]
        ys = [-v['anytime_loss_auc'] for v in summaries]
        std = lambda vs: statistics.stdev(vs) if len(vs)>1 else 0.
        ax.errorbar(statistics.mean(xs),statistics.mean(ys),xerr=std(xs),yerr=std(ys),fmt='o',label=arm+'/'+width)
    ax.set(xlabel='Mean last-50 online error (regression stability proxy)',
           ylabel='Negative task loss AUC (regression plasticity proxy)')
    if groups: ax.legend(fontsize=6)
    fig.savefig(figure,bbox_inches='tight'); plt.close(fig)
else:
    stability_plasticity(records.values(),figure)
from IPython.display import SVG, display
display(SVG(filename=str(figure)))
for path in ROOT.rglob('best_config.json'): print(path, path.read_text())
for path in ROOT.rglob('reproduction_gate.json'): print(path, path.read_text())
archive_outputs()
if session_paused.is_set(): print('PAUSED: attach output archive and rerun to resume')
if archive_errors: raise RuntimeError(archive_errors)
